
# Wildfire × Air Quality dataset builder

This notebook pulls **daily** air quality metrics from EPA **AQS Data Mart**, merges in **hourly weather** from **Open‑Meteo (Historical)** to daily means, and tags each day with **NOAA HMS Smoke** presence (light/medium/heavy) at each monitor location.

### Output columns
- `date` (YYYY-MM-DD)
- `site_id` (AQS site code)
- `latitude`, `longitude`
- `pm25` (µg/m³)
- `pm10` (µg/m³)
- `aqi` (daily **max** AQI across pollutants)
- `temperature_f` (°F, daily mean 2m temp)
- `humidity_percent` (% RH, daily mean 2m)
- `wind_speed_mph` (mph, daily mean 10m wind)
- `no2_ppb` (ppb)
- `o3_ppb` (ppb)
- `co_ppm` (ppm)
- `status` (HMS smoke class at site/day: `none|light|medium|heavy`)

> Notes:  
> • AQS requires **free registration** for an `email` and `key`.  
> • HMS smoke polygons are daily KML files.  
> • Open‑Meteo needs no key; we use the historical ERA5 endpoint.  
> • Time is handled in **UTC** internally; HMS “Date” is UTC.  
> • For AQI we take the **max** pollutant‑specific AQI reported by AQS per site/day; if missing we compute it from concentrations using EPA breakpoints.


In [106]:

# If running locally, uncomment to install deps (skip if your env already has them).
# %pip install pandas requests fastkml shapely pyproj tqdm python-dateutil python-dotenv requests-cache

import os, io, math, json, time, zipfile, datetime as dt
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from pathlib import Path

import pandas as pd
import requests
import requests_cache
from dateutil import tz
from tqdm import tqdm
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# KML parsing & geometry
from fastkml import kml
from shapely.geometry import shape, Point, Polygon, MultiPolygon
from shapely.ops import unary_union

# Setup caching for API calls (expires after 7 days by default)
CACHE_DIR = Path("./cache")
CACHE_DIR.mkdir(exist_ok=True)

# Initialize requests cache with SQLite backend
requests_cache.install_cache(
    cache_name=str(CACHE_DIR / "api_cache"),
    backend="sqlite",
    expire_after=604800,  # 7 days in seconds
    allowable_codes=[200],
    allowable_methods=["GET"],
)

print(f"✓ API caching enabled: {CACHE_DIR / 'api_cache.sqlite'}")
print(f"  Cache expires after: 7 days")
print(f"  To clear cache, delete the cache/ directory or run requests_cache.clear()")


✓ API caching enabled: cache/api_cache.sqlite
  Cache expires after: 7 days
  To clear cache, delete the cache/ directory or run requests_cache.clear()


## Cache Management (Optional)

Run this cell if you need to clear the cache or check cache statistics.

In [107]:

# Check cache stats
cache_info = requests_cache.get_cache()
print(f"Cache location: {CACHE_DIR / 'api_cache.sqlite'}")
print(f"Cached responses: {len(cache_info.responses)}")
print(f"Cache size: {(CACHE_DIR / 'api_cache.sqlite').stat().st_size / 1024 / 1024:.2f} MB")

# Uncomment to clear the cache:
# requests_cache.clear()
# print("✓ Cache cleared!")



Cache location: cache/api_cache.sqlite
Cached responses: 117
Cache size: 22.48 MB


## Configure your run


In [ ]:

# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# REQUIRED: put your AQS credentials here (register at https://aqs.epa.gov/)
AQS_EMAIL = os.getenv("AQS_EMAIL", "your_email@example.com")
AQS_KEY   = os.getenv("AQS_KEY", "gg")

# Area & dates (example: Southern California)
BBOX = (-121.25, 32.53, -114.13, 35.75)  # (min_lon, min_lat, max_lon, max_lat)
START_DATE = "2024-01-15"            # inclusive
END_DATE   = "2024-10-15"            # inclusive

# How to aggregate Open-Meteo hourly -> daily
WEATHER_AGG = "mean"  # "mean" or "median"
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

# Pollutants (AQS parameter codes)
PARAM_CODES = {
    "pm25": "88101",  # PM2.5 LC (µg/m3)
    "pm10": "81102",  # PM10 - LC (µg/m3)
    "o3":   "44201",  # Ozone
    "no2":  "42602",  # NO2
    "co":   "42101"   # CO
}


## AQI computation helpers (fallback if missing in AQS response)

In [109]:

# AQI breakpoints (US EPA, 2018 tech doc). Units:
# PM2.5, PM10 in µg/m3; CO in ppm; O3 in ppb (8h), NO2 in ppb (1h surrogate for index).
# O3: use 8h averages; at daily scale we assume daily max 8h is reflected by AQS 'aqi' for O3.
# If missing, we estimate from daily mean as a fallback (approximation).

AQI_BREAKPOINTS = {
    "pm25": [
        (0.0, 12.0, 0, 50),
        (12.1, 35.4, 51, 100),
        (35.5, 55.4, 101, 150),
        (55.5, 150.4, 151, 200),
        (150.5, 250.4, 201, 300),
        (250.5, 350.4, 301, 400),
        (350.5, 500.4, 401, 500),
    ],
    "pm10": [
        (0, 54, 0, 50),
        (55, 154, 51, 100),
        (155, 254, 101, 150),
        (255, 354, 151, 200),
        (355, 424, 201, 300),
        (425, 504, 301, 400),
        (505, 604, 401, 500),
    ],
    "o3_8h_ppb": [
        (0, 54, 0, 50),
        (55, 70, 51, 100),
        (71, 85, 101, 150),
        (86, 105, 151, 200),
        (106, 200, 201, 300),
    ],
    "co_ppm": [
        (0.0, 4.4, 0, 50),
        (4.5, 9.4, 51, 100),
        (9.5, 12.4, 101, 150),
        (12.5, 15.4, 151, 200),
        (15.5, 30.4, 201, 300),
        (30.5, 40.4, 301, 400),
        (40.5, 50.4, 401, 500),
    ],
    "no2_ppb": [
        (0, 53, 0, 50),
        (54, 100, 51, 100),
        (101, 360, 101, 150),
        (361, 649, 151, 200),
        (650, 1249, 201, 300),
        (1250, 1649, 301, 400),
        (1650, 2049, 401, 500),
    ]
}

def compute_aqi_from_bp(x, bp_list):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None
    for (Clow, Chigh, Ilow, Ihigh) in bp_list:
        if Clow <= x <= Chigh:
            return round((Ihigh - Ilow) / (Chigh - Clow) * (x - Clow) + Ilow)
    return None

def estimate_aqi_row(row):
    aqi_candidates = []
    if pd.notna(row.get("pm25")):
        aqi_candidates.append(compute_aqi_from_bp(row["pm25"], AQI_BREAKPOINTS["pm25"]))
    if pd.notna(row.get("pm10")):
        aqi_candidates.append(compute_aqi_from_bp(row["pm10"], AQI_BREAKPOINTS["pm10"]))
    if pd.notna(row.get("o3_ppb")):
        aqi_candidates.append(compute_aqi_from_bp(row["o3_ppb"], AQI_BREAKPOINTS["o3_8h_ppb"]))
    if pd.notna(row.get("co_ppm")):
        aqi_candidates.append(compute_aqi_from_bp(row["co_ppm"], AQI_BREAKPOINTS["co_ppm"]))
    if pd.notna(row.get("no2_ppb")):
        aqi_candidates.append(compute_aqi_from_bp(row["no2_ppb"], AQI_BREAKPOINTS["no2_ppb"]))
    aqi_candidates = [a for a in aqi_candidates if a is not None]
    return max(aqi_candidates) if aqi_candidates else None


## Fetch AQS daily data (by bounding box)

In [110]:

AQS_BASE = "https://aqs.epa.gov/data/api"

def aqs_query_daily_by_box(bdate: str, edate: str, bbox: Tuple[float, float, float, float], params: List[str]):
    minlon, minlat, maxlon, maxlat = bbox
    url = f"{AQS_BASE}/dailyData/byBox"
    q = {
        "email": AQS_EMAIL, "key": AQS_KEY,
        "param": ",".join(params),
        "bdate": bdate.replace("-", ""),
        "edate": edate.replace("-", ""),
        "minlat": minlat, "minlon": minlon, "maxlat": maxlat, "maxlon": maxlon
    }
    r = requests.get(url, params=q, timeout=120)
    r.raise_for_status()
    js = r.json()
    if "Data" not in js:
        raise RuntimeError(f"AQS returned no Data: {js}")
    df = pd.DataFrame(js["Data"])
    return df

def normalize_units(df: pd.DataFrame) -> pd.DataFrame:
    # Keep only necessary columns (only include columns that exist)
    required = ["date_local","site_number","parameter","parameter_code","arithmetic_mean","latitude","longitude"]
    optional = ["unit","aqi","state_code","county_code"]
    keep = required + [col for col in optional if col in df.columns]
    df = df[keep].copy()
    
    # Add unit column if missing
    if "unit" not in df.columns:
        df["unit"] = None
    
    # Standardize pollutant keys
    code_map = {v:k for k,v in PARAM_CODES.items()}
    df["pollutant"] = df["parameter_code"].map(code_map)
    
    # Convert to required units
    def convert(row):
        val = row["arithmetic_mean"]
        unit = row.get("unit", None)
        pol  = row["pollutant"]
        if pd.isna(val): 
            return None
        # Concentration unit conversions
        if pol in ("pm25","pm10"):
            # Expect µg/m3 already
            return float(val)
        if pol in ("o3","no2"):
            # We want ppb
            if unit and "Parts per million" in str(unit):
                return float(val) * 1000.0
            elif unit and "Parts per billion" in str(unit):
                return float(val)
            else:
                return float(val)  # fallback
        if pol == "co":
            # We want ppm
            if unit and "Parts per million" in str(unit):
                return float(val)
            elif unit and "Parts per billion" in str(unit):
                return float(val) / 1000.0
            else:
                return float(val)
        return float(val)

    df["value_std"] = df.apply(convert, axis=1)
    # Pivot pollutants to columns
    pivot = df.pivot_table(index=["date_local","site_number","latitude","longitude"],
                           columns="pollutant",
                           values="value_std",
                           aggfunc="mean").reset_index()
    # AQI: take maximum across pollutants using reported AQS 'aqi' when available
    aqi_df = df.copy()
    has_aqi = "aqi" in df.columns
    if has_aqi:
        aqi_df["aqi_num"] = pd.to_numeric(aqi_df["aqi"], errors="coerce")
        aqi_max = aqi_df.groupby(["date_local","site_number"])["aqi_num"].max().reset_index().rename(columns={"aqi_num":"aqi"})
        out = pivot.merge(aqi_max, on=["date_local","site_number"], how="left")
    else:
        out = pivot
        out["aqi"] = None
    # Rename columns to final names
    out = out.rename(columns={
        "date_local":"date",
        "pm25":"pm25",
        "pm10":"pm10",
        "o3":"o3_ppb",
        "no2":"no2_ppb",
        "co":"co_ppm"
    })
    return out

# Example call (commented out to avoid running on import):
# df_raw = aqs_query_daily_by_box(START_DATE, END_DATE, BBOX, list(PARAM_CODES.values()))
# aqs_daily = normalize_units(df_raw)
# aqs_daily.head()


## Fetch hourly weather from Open‑Meteo (archive) and aggregate to daily

In [ ]:

OM_BASE = "https://archive-api.open-meteo.com/v1/era5"

def get_weather_for_point(lat: float, lon: float, start_date: str, end_date: str) -> pd.DataFrame:
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start_date, "end_date": end_date,
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m",
        "temperature_unit": "fahrenheit",
        "windspeed_unit": "mph",
        "timezone": "UTC"
    }
    r = requests.get(OM_BASE, params=params, timeout=1000)
    r.raise_for_status()
    js = r.json()
    # Convert to hourly DataFrame
    hours = pd.DataFrame(js["hourly"])
    hours["time"] = pd.to_datetime(hours["time"]).dt.tz_localize("UTC")
    # Aggregate
    if WEATHER_AGG == "median":
        daily = hours.groupby(hours["time"].dt.date).median(numeric_only=True)
    else:
        daily = hours.groupby(hours["time"].dt.date).mean(numeric_only=True)
    daily.index = pd.to_datetime(daily.index)
    daily = daily.rename(columns={
        "temperature_2m":"temperature_f",
        "relative_humidity_2m":"humidity_percent",
        "wind_speed_10m":"wind_speed_mph"
    })
    daily = daily.reset_index().rename(columns={"time":"date"})
    daily["date"] = daily["date"].dt.strftime("%Y-%m-%d")
    return daily

def attach_weather(aqs_daily: pd.DataFrame) -> pd.DataFrame:
    rows = []
    # Unique site coordinates
    sites = aqs_daily[["site_number","latitude","longitude"]].drop_duplicates()
    # Cache weather per site
    weather_cache: Dict[str, pd.DataFrame] = {}
    for _, s in tqdm(sites.iterrows(), total=len(sites), desc="Fetching weather"):
        key = f"{s.latitude:.4f}_{s.longitude:.4f}"
        if key not in weather_cache:
            weather_cache[key] = get_weather_for_point(float(s.latitude), float(s.longitude), START_DATE, END_DATE)
        wdf = weather_cache[key].copy()
        wdf["site_number"] = s.site_number
        rows.append(wdf)
    w_all = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    out = aqs_daily.merge(w_all, on=["site_number","date"], how="left")
    return out


## Tag each site/day with NOAA HMS Smoke polygons

In [112]:

HMS_BASE = "https://satepsanone.nesdis.noaa.gov/pub/FIRE/web/HMS/Smoke_Polygons/KML"

DENSITY_ORDER = {"none":0, "light":1, "medium":2, "heavy":3}

def hms_kml_url_for_date(d: dt.date) -> str:
    y = d.strftime("%Y"); m = d.strftime("%m"); ds = d.strftime("%Y%m%d")
    return f"{HMS_BASE}/{y}/{m}/hms_smoke{ds}.kml"

def load_hms_polys_for_date(d: dt.date) -> List[Tuple[str, MultiPolygon]]:
    url = hms_kml_url_for_date(d)
    try:
        r = requests.get(url, timeout=120)
        r.raise_for_status()
    except Exception as e:
        return []  # no file available (smoke-free day or missing)
    k = kml.KML()
    k.from_string(r.content)
    polys = []
    def iter_features(obj):
        features_attr = getattr(obj, "features", None)
        if features_attr is None:
            return []
        if callable(features_attr):
            try:
                return list(features_attr())
            except TypeError:
                return list(features_attr)
        try:
            return list(features_attr)
        except TypeError:
            return []
    def walk(feats):
        for f in feats:
            children = iter_features(f)
            if children:
                yield from walk(children)
                continue
            yield f
    for f in walk(iter_features(k)):
        # Density in ExtendedData if present; fallback to folder name
        density = "none"
        try:
            if hasattr(f, "extended_data") and f.extended_data:
                for data in f.extended_data.elements:
                    if getattr(data, "name", "").lower() == "density":
                        density = str(data.value).strip().lower()
            if density not in DENSITY_ORDER:
                density = "none"
            geom = f.geometry
            if geom is None:
                continue
            g = shape(geom)
            if isinstance(g, (Polygon, MultiPolygon)):
                polys.append((density, g if isinstance(g, MultiPolygon) else MultiPolygon([g])))
        except Exception:
            continue
    return polys

def tag_smoke_status(aqs_daily: pd.DataFrame) -> pd.DataFrame:
    dates = sorted(aqs_daily["date"].unique())
    # Preload HMS polygons
    poly_map: Dict[str, List[Tuple[str, MultiPolygon]]] = {}
    for ds in tqdm(dates, desc="Downloading HMS KML"):
        d = dt.datetime.strptime(ds, "%Y-%m-%d").date()
        poly_map[ds] = load_hms_polys_for_date(d)

    def classify_point(row):
        pt = Point(float(row["longitude"]), float(row["latitude"]))
        best = "none"
        for density, geom in poly_map.get(row["date"], []):
            try:
                if geom.contains(pt) or geom.touches(pt):
                    if DENSITY_ORDER[density] > DENSITY_ORDER[best]:
                        best = density
            except Exception:
                continue
        return best

    aqs_daily["status"] = aqs_daily.apply(classify_point, axis=1)
    return aqs_daily


## Run the pipeline & save CSV

In [113]:

def run_pipeline():
    print("Fetching AQS daily data ...")
    df_raw = aqs_query_daily_by_box(START_DATE, END_DATE, BBOX, list(PARAM_CODES.values()))
    print(f"AQS rows: {len(df_raw):,}")
    aqs_daily = normalize_units(df_raw)

    # Ensure final columns exist
    for col in ["pm25","pm10","o3_ppb","no2_ppb","co_ppm"]:
        if col not in aqs_daily.columns:
            aqs_daily[col] = pd.NA

    # Attach weather
    enriched = attach_weather(aqs_daily)

    # Fill AQI where missing
    if "aqi" not in enriched.columns:
        enriched["aqi"] = None
    enriched["aqi"] = pd.to_numeric(enriched["aqi"], errors="coerce")
    need_aqi = enriched["aqi"].isna()
    if need_aqi.any():
        enriched.loc[need_aqi, "aqi"] = enriched[need_aqi].apply(estimate_aqi_row, axis=1)

    # Tag smoke
    labelled = tag_smoke_status(enriched)

    # Reorder columns
    cols = ["date","site_number","latitude","longitude",
            "pm25","pm10","aqi","temperature_f","humidity_percent","wind_speed_mph",
            "no2_ppb","o3_ppb","co_ppm","status"]
    labelled = labelled[cols].sort_values(["date","site_number"])

    # Save
    out_dir = Path.cwd() / "data"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "wildfire_aqi_dataset.csv"
    labelled.to_csv(out_path, index=False)
    print(f"Saved: {out_path}  (rows={len(labelled):,})")
    return labelled

# To execute, uncomment the next two lines:
df_out = run_pipeline()
df_out.head()


Fetching AQS daily data ...


ReadTimeout: HTTPSConnectionPool(host='aqs.epa.gov', port=443): Read timed out. (read timeout=120)


## Tips & caveats

- **AQS API limits**: large date ranges must be split by year. If you hit errors, loop by calendar year.  
- **AQS missing days**: recent days can be provisional; sometimes values are missing until QA/QC.  
- **Units**: the code normalizes to pm25/pm10 in µg/m³, NO₂ & O₃ in **ppb**, CO in **ppm**.  
- **O₃ AQI** requires 8‑hour max; AQS provides pollutant‑specific `aqi` which we use if available; otherwise we estimate from daily mean as a fallback (approximate).  
- **HMS polygons** represent analyst‑drawn smoke **extent** for the day (UTC). Presence inside a polygon indicates potential smoke influence, **not** necessarily surface smoke.  
- You can filter to `status != 'none'` to get smoke‑impacted days.
